## Setup

In [2]:
import wntr
import numpy as np
import pandas as pd

In [3]:
from wntr.epanet.util import to_si, FlowUnits, HydParam

wn = wntr.network.WaterNetworkModel(
    r"C:\Data\pipeline-leakage-system\Phase 1\EPANET\Experiment_network.inp"
)

EMITTER_C_TO_SI = to_si(FlowUnits.GPM, 1.0, HydParam.EmitterCoeff)
GPM_TO_M3S = 0.0000630902
FT_TO_M = 0.3048

In [4]:
print(wn.node_name_list)
print(wn.link_name_list)

for name in wn.junction_name_list:
    print(name, wn.get_node(name))

for name in wn.link_name_list:
    p = wn.get_link(name)
    print(name, p.length, p.diameter, p.roughness, p.initial_status)

['J1', 'J2', 'R1']
['P1', 'P2']
J1 J1
J2 J2
P1 999.999999996 0.2999999999988 100.0 Open
P2 799.9999999967999 0.24999999999899997 100.0 Open


In [ ]:
import os

path = r"C:\Data\pipeline-leakage-system\Phase 1\EPANET\Experiment_network.inp"

print("Exists:", os.path.exists(path))
print("File:", path)

Exists: True
File: C:\Data\pipeline-leakage-system\Phase 1\EPANET\Experiment_network.inp


In [6]:
# CELL 3 — inspect the new network structure

print("Nodes:", wn.node_name_list)
print("Links:", wn.link_name_list)

print("\nJunctions:", wn.junction_name_list)
print("Reservoirs:", wn.reservoir_name_list)
print("Tanks:", wn.tank_name_list)
print("Pipes:", wn.pipe_name_list)
print("Pumps:", wn.pump_name_list)
print("Valves:", wn.valve_name_list)

Nodes: ['J1', 'J2', 'R1']
Links: ['P1', 'P2']

Junctions: ['J1', 'J2']
Reservoirs: ['R1']
Tanks: []
Pipes: ['P1', 'P2']
Pumps: []
Valves: []


### Create the network

In [7]:
wn = wntr.network.WaterNetworkModel()

# Nodes
wn.add_reservoir("R1", base_head=100)
wn.add_junction("J1", base_demand=10)
wn.add_junction("J2", base_demand=8)

# Pipes
wn.add_pipe("P1", "R1", "J1", length=1000, diameter=0.30, roughness=100)
wn.add_pipe("P2", "J1", "J2", length=800, diameter=0.25, roughness=100)

print("Nodes:", wn.node_name_list)
print("Links:", wn.link_name_list)
print("Junctions:", wn.junction_name_list)
print("Reservoirs:", wn.reservoir_name_list)
print("Pipes:", wn.pipe_name_list)

Nodes: ['R1', 'J1', 'J2']
Links: ['P1', 'P2']
Junctions: ['J1', 'J2']
Reservoirs: ['R1']
Pipes: ['P1', 'P2']


### Write your network into the file

In [8]:
wn.write_inpfile(
    r"C:\Data\pipeline-leakage-system\Phase 1\EPANET\Experiment_network.inp"
)

AttributeError: 'WaterNetworkModel' object has no attribute 'write_inpfile'

In [9]:
wntr.network.write_inpfile(
    wn,
    r"C:\Data\pipeline-leakage-system\Phase 1\EPANET\Experiment_network.inp"
)

Unless you perform this step, changes in the network exist only in the WNTR object in memory. This step makes the changes in-place, inside the file

### Verify that the `.inp` file was written and can be loaded correctly

In [ ]:
path = r"C:\Data\pipeline-leakage-system\Phase 1\EPANET\Experiment_network.inp"

test_wn = wntr.network.WaterNetworkModel(path)

print("Nodes:", test_wn.node_name_list)
print("Links:", test_wn.link_name_list)
print("Junctions:", test_wn.junction_name_list)
print("Reservoirs:", test_wn.reservoir_name_list)
print("Pipes:", test_wn.pipe_name_list)
print("Valves:", test_wn.valve_name_list)
print("Tanks:", test_wn.tank_name_list)
print("Pumps:", test_wn.pump_name_list)

print("\nPipe details:")
for name in test_wn.pipe_name_list:
    p = test_wn.get_link(name)
    print(name, ":", p.start_node_name, "->", p.end_node_name,
          "| L =", p.length, "| D =", p.diameter, "| C =", p.roughness)

Nodes: ['J1', 'J2', 'R1']
Links: ['P1', 'P2']
Junctions: ['J1', 'J2']
Reservoirs: ['R1']
Pipes: ['P1', 'P2']

Pipe details:
P1 : R1 -> J1 | L = 999.999999996 | D = 0.2999999999988 | C = 100.0
P2 : J1 -> J2 | L = 799.9999999967999 | D = 0.24999999999899997 | C = 100.0


## Variations

#### A small **topology variation**: add a third junction and a branch pipe from `J1`.

**Variation:** the previous network was a simple chain `R1 → J1 → J2`; now `J1` branches into both `J2` and `J3`.

In [11]:
# Add one junction
wn.add_junction("J3", base_demand=6)

# Add one branch
wn.add_pipe("P3", "J1", "J3", length=600, diameter=0.20, roughness=100)

print("Nodes:", wn.node_name_list)
print("Links:", wn.link_name_list)

Nodes: ['R1', 'J1', 'J2', 'J3']
Links: ['P1', 'P2', 'P3']


#### Let's make the next variation a **loop**: connect `J3` back to `J2`.
**Variation:** the network now contains a closed loop: `J1 → J2 → J3 → J1`.

In [ ]:
wn.add_pipe(
    "P4",
    "J3",
    "J2",
    length=500,
    diameter=0.20,
    roughness=100
)

print("Nodes:", wn.node_name_list)
print("Links:", wn.link_name_list)

Nodes: ['R1', 'J1', 'J2', 'J3']
Links: ['P1', 'P2', 'P3', 'P4']


#### Next, let's vary the **pipe properties** rather than the topology:
**Variation:** same topology, but `P3` now has different length, diameter, and roughness.

In [13]:
p3 = wn.get_link("P3")

p3.length = 1200
p3.diameter = 0.30
p3.roughness = 110

print(
    "P3:",
    "L =", p3.length,
    "| D =", p3.diameter,
    "| C =", p3.roughness
)

P3: L = 1200.0 | D = 0.3 | C = 110.0


#### Next, let's vary the **demand** at a junction:
**Variation:** the network topology and pipe properties stay the same; only the water demand at `J3` changes.

In [14]:
j3 = wn.get_node("J3")
j3.demand_timeseries_list[0].base_value = 12

print("J3 demand:", j3.demand_timeseries_list[0].base_value)

J3 demand: 12.0


means J3's base demand is set to 12 (in the network's configured flow units)

#### Next, let's introduce a **second reservoir**—another source of water/head in the network:
**Variation:** the network now has **two supply sources**, `R1` and `R2`, instead of a single reservoir.

In [15]:
wn.add_reservoir("R2", base_head=95)

# Connect it to J3
wn.add_pipe(
    "P5",
    "R2",
    "J3",
    length=700,
    diameter=0.25,
    roughness=100
)

print("Nodes:", wn.node_name_list)
print("Links:", wn.link_name_list)
print("Reservoirs:", wn.reservoir_name_list)

Nodes: ['R1', 'J1', 'J2', 'J3', 'R2']
Links: ['P1', 'P2', 'P3', 'P4', 'P5']
Reservoirs: ['R1', 'R2']


#### Next, let's add a **tank** to introduce storage dynamics:
**Variation:** the network now contains a **storage tank** connected to the existing network.

In [16]:
wn.add_tank(
    "T1",
    elevation=80,
    init_level=10,
    min_level=0,
    max_level=20,
    diameter=10
)

# Connect J2 to the tank
wn.add_pipe(
    "P6",
    "J2",
    "T1",
    length=500,
    diameter=0.20,
    roughness=100
)

print("Tanks:", wn.tank_name_list)
print("Links:", wn.link_name_list)

Tanks: ['T1']
Links: ['P1', 'P2', 'P3', 'P4', 'P5', 'P6']


#### Next useful variation is a **pump**, introducing an active component that changes hydraulic head:
**Variation:** `R1 → J1` now has an active pump rather than relying only on passive pipe flow.

In [17]:
wn.add_pump(
    "PU1",
    "R1",
    "J1",
    pump_type="POWER",
    pump_parameter=50
)

print("Pumps:", wn.pump_name_list)
print("Links:", wn.link_name_list)

Pumps: ['PU1']
Links: ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'PU1']


#### Next, let's introduce a **valve**
**Variation:** the network now contains a controllable hydraulic element between `J2` and `J3`.

In [18]:
wn.add_valve(
    "V1",
    "J2",
    "J3",
    diameter=0.20,
    valve_type="PRV",
    minor_loss=0
)

print("Valves:", wn.valve_name_list)
print("Links:", wn.link_name_list)

Valves: ['V1']
Links: ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'PU1', 'V1']


#### For opening/closing a valve

In [19]:
wn.get_link("V1").initial_status = "OPEN"
wn.get_link("V1").initial_status = "CLOSED"

#### Next, let's introduce **leakage** explicitly using an emitter at `J3`:
**Variation:** `J3` now has a leak represented by an emitter.

In [20]:
j3 = wn.get_node("J3")
j3.emitter_coefficient = 1.0

print("J3 emitter coefficient:", j3.emitter_coefficient)

J3 emitter coefficient: 1.0


#### Next, let's vary the **leak location** by adding an emitter to a different junction:
**Variation:** the leak is now at `J2` along with in `J3`.

In [21]:
j2 = wn.get_node("J2")
j2.emitter_coefficient = 0.8

print("J2 emitter coefficient:", j2.emitter_coefficient)

J2 emitter coefficient: 0.8


#### changing a pipe's status (open/closed)

In [22]:
p4 = wn.get_link("P4")
p4.initial_status = "CLOSED"

print("P4 status:", p4.initial_status)

P4 status: Closed


#### add a second demand pattern to the same junction
so demand can change over time rather than staying constant.
This introduces **temporal operating variation**

In [23]:
pattern = wn.add_pattern("J3_pattern", [0.8, 1.0, 1.2])

j3 = wn.get_node("J3")
j3.demand_timeseries_list[0].pattern_name = "J3_pattern"

print("J3 base demand:", j3.demand_timeseries_list[0].base_value)
print("J3 pattern:", j3.demand_timeseries_list[0].pattern_name)

J3 base demand: 12.0
J3 pattern: J3_pattern


#### Next, let's try a **pump curve** instead of the fixed-power pump. This is more realistic for an actual pipeline system:
**Variation:** pump head now depends on the **flow rate**, rather than being represented by a fixed power value.

In [24]:
wn.add_curve(
    "PumpCurve",
    "HEAD",
    [(0, 30), (0.01, 25), (0.02, 15), (0.03, 0)]
)

wn.get_link("PU1").pump_type = "HEAD"
wn.get_link("PU1").pump_parameter = "PumpCurve"

print("Pump type:", wn.get_link("PU1").pump_type)
print("Pump curve:", wn.get_link("PU1").pump_parameter)

AttributeError: property 'pump_type' of 'PowerPump' object has no setter

the error is because `PU1` was created as a **`PowerPump`**, and `pump_type` cannot be changed after creation.

So that cell was wrong. Don't modify `PU1`; create a new pump with a head curve instead

We can use `PU2` for this experiment while leaving `PU1` unchanged.

In [25]:
wn.add_pump(
    "PU2",
    "R1",
    "J1",
    pump_type="HEAD",
    pump_parameter=[(0, 30), (0.01, 25), (0.02, 15), (0.03, 0)]
)

print("Pumps:", wn.pump_name_list)

ValueError: pump_parameter must be a float, convertible to float, or a string. Received [(0, 30), (0.01, 25), (0.02, 15), (0.03, 0)] of type list

`add_pump()` expects `pump_parameter` to be a **number or curve name**, not the curve data itself.

Create the curve first, then pass its **name**:

The important distinction is: `pump_parameter="PumpCurve"` → tells WNTR **which curve to use**.

In [26]:
wn.add_curve(
    "PumpCurve",
    "HEAD",
    [(0, 30), (0.01, 25), (0.02, 15), (0.03, 0)]
)

# Create pump using the curve name
wn.add_pump(
    "PU2",
    "R1",
    "J1",
    pump_type="HEAD",
    pump_parameter="PumpCurve"
)

print("Pumps:", wn.pump_name_list)

Pumps: ['PU1', 'PU2']


#### Next, let's introduce **different elevations** for the junctions. 
This affects hydraulic head without changing topology

**Variation:** the network now represents terrain with different elevations at different junctions.

In [27]:
# Assign different elevations
wn.get_node("J1").elevation = 100
wn.get_node("J2").elevation = 90
wn.get_node("J3").elevation = 75

print("J1 elevation:", wn.get_node("J1").elevation)
print("J2 elevation:", wn.get_node("J2").elevation)
print("J3 elevation:", wn.get_node("J3").elevation)

J1 elevation: 100
J2 elevation: 90
J3 elevation: 75


## Clear the `.inp` file completely

In [28]:
path = r"C:\Data\pipeline-leakage-system\Phase 1\EPANET\Experiment_network.inp"
open(path, "w").close()

#### Check if the network was removed or not

In [29]:
path = r"C:\Data\pipeline-leakage-system\Phase 1\EPANET\Experiment_network.inp"

test_wn = wntr.network.WaterNetworkModel(path)

print("Nodes:", test_wn.node_name_list)
print("Links:", test_wn.link_name_list)
print("Junctions:", test_wn.junction_name_list)
print("Reservoirs:", test_wn.reservoir_name_list)
print("Pipes:", test_wn.pipe_name_list)
print("Valves:", test_wn.valve_name_list)
print("Tanks:", test_wn.tank_name_list)
print("Pumps:", test_wn.pump_name_list)

print("\nPipe details:")
for name in test_wn.pipe_name_list:
    p = test_wn.get_link(name)
    print(name, ":", p.start_node_name, "->", p.end_node_name,
          "| L =", p.length, "| D =", p.diameter, "| C =", p.roughness)

Nodes: []
Links: []
Junctions: []
Reservoirs: []
Pipes: []
Valves: []
Tanks: []
Pumps: []

Pipe details:
